# 🚀 HungerHub Oracle Data Extraction - Optimal Strategy (Full Production Data)

## 📊 **Extraction Strategy Based on Performance Analysis**
- **Method**: Sequential Extraction (Proven optimal at **991.8 rows/sec**)
- **Target**: High-priority tables from comprehensive database survey
- **Total Dataset**: 7.6M+ rows across 283 tables
- **Estimated Time**: ~45 minutes parallel, ~2.3 hours sequential

## 🎯 **Database Survey Results**
- **Total Tables Surveyed**: 283
- **Tables with Data**: 178
- **High Priority Tables**: 5 (451K rows, ~8min extraction)
- **Total Data Volume**: ~2.2 GB

## 🔴 **High Priority Tables for POC**
| Table | Rows | Columns | Size | Est. Time |
|-------|------|---------|------|-----------|
| RW_ORDER_ITEM | 230,282 | 75 | 72MB | ~4.0min |
| RW_ORDER_SUPPLIER | 96,552 | 7 | 7MB | ~1.7min |
| RW_PURCHASE_ORDER | 96,552 | 51 | 22MB | ~1.7min |
| AMX_DONATION_LINES | 27,099 | 26 | 320MB | ~0.5min |
| AMX_DONATION_HEADER | 1,389 | 30 | 59MB | ~0.0min |

---

In [1]:
# 🛠️ Section 1: Environment Setup and Optimized Imports
import os
import sys
import pandas as pd
import numpy as np
import cx_Oracle
import json
import logging
from datetime import datetime, timedelta
from pathlib import Path
from concurrent.futures import ThreadPoolExecutor, as_completed
import threading
import time
import gc
from dotenv import load_dotenv
import warnings
warnings.filterwarnings('ignore')

# Performance monitoring
import psutil

# Load environment configuration
config_path = '/home/tagazureuser/cgorr/2week_poc_execution/hungerhub_poc/config/.env'
load_dotenv(config_path)

# Create output directories
output_base = Path('optimized_extraction_output')
output_dirs = {
    'csv': output_base / 'csv',
    'parquet': output_base / 'parquet', 
    'logs': output_base / 'logs',
    'metadata': output_base / 'metadata'
}

for dir_path in output_dirs.values():
    dir_path.mkdir(parents=True, exist_ok=True)

print("✅ Environment setup completed")
print(f"📁 Output directories created: {list(output_dirs.keys())}")
print(f"💾 Data will be saved to: {output_base}")

✅ Environment setup completed
📁 Output directories created: ['csv', 'parquet', 'logs', 'metadata']
💾 Data will be saved to: optimized_extraction_output


In [2]:
# 📊 Section 2: Database Survey Results and High Priority Tables

# High priority tables identified from database survey
HIGH_PRIORITY_TABLES = [
    {
        'name': 'AMX_DONATION_LINES',
        'full_name': 'RWTXN_46.AMX_DONATION_LINES',
        'rows': 27099,
        'columns': 26,
        'size_mb': 320.0,
        'estimated_minutes': 0.5,
        'business_priority': 'critical',
        'description': 'Core donation line items data'
    },
    {
        'name': 'AMX_DONATION_HEADER', 
        'full_name': 'RWTXN_46.AMX_DONATION_HEADER',
        'rows': 1389,
        'columns': 30,
        'size_mb': 59.0,
        'estimated_minutes': 0.0,
        'business_priority': 'critical',
        'description': 'Donation header/summary data'
    },
    {
        'name': 'RW_ORDER_ITEM',
        'full_name': 'RWTXN_46.RW_ORDER_ITEM', 
        'rows': 230282,
        'columns': 75,
        'size_mb': 72.0,
        'estimated_minutes': 4.0,
        'business_priority': 'high',
        'description': 'Order line items and procurement data'
    },
    {
        'name': 'RW_ORDER_SUPPLIER',
        'full_name': 'RWTXN_46.RW_ORDER_SUPPLIER',
        'rows': 96552,
        'columns': 7, 
        'size_mb': 7.0,
        'estimated_minutes': 1.7,
        'business_priority': 'high',
        'description': 'Order-supplier relationship data'
    },
    {
        'name': 'RW_PURCHASE_ORDER',
        'full_name': 'RWTXN_46.RW_PURCHASE_ORDER',
        'rows': 96552,
        'columns': 51,
        'size_mb': 22.0, 
        'estimated_minutes': 1.7,
        'business_priority': 'high',
        'description': 'Purchase order header data'
    }
]

# Additional medium priority tables for comprehensive analysis
MEDIUM_PRIORITY_TABLES = [
    {
        'name': 'AMX_OFFER_HEADER',
        'full_name': 'RWTXN_46.AMX_OFFER_HEADER',
        'rows': 26343,
        'columns': 46,
        'estimated_minutes': 0.5
    },
    {
        'name': 'AMX_OFFER_LINES', 
        'full_name': 'RWTXN_46.AMX_OFFER_LINES',
        'rows': 91783,
        'columns': 32,
        'estimated_minutes': 1.6
    },
    {
        'name': 'RW_ORG',
        'full_name': 'RWTXN_46.RW_ORG', 
        'rows': 630,
        'columns': 44,
        'estimated_minutes': 0.0
    },
    {
        'name': 'RW_USER',
        'full_name': 'RWTXN_46.RW_USER',
        'rows': 5970,
        'columns': 37,
        'estimated_minutes': 0.1
    }
]

# Calculate totals
total_high_priority_rows = sum(t['rows'] for t in HIGH_PRIORITY_TABLES)
total_high_priority_time = sum(t['estimated_minutes'] for t in HIGH_PRIORITY_TABLES)

print("🔴 High Priority Tables (Critical for POC):")
for table in HIGH_PRIORITY_TABLES:
    print(f"   📋 {table['name']:<25} {table['rows']:>8,} rows, {table['columns']:>2} cols, {table['size_mb']:>5.1f}MB (~{table['estimated_minutes']:>4.1f}min)")
    
print(f"\n📊 High Priority Summary:")
print(f"   Total Rows: {total_high_priority_rows:,}")
print(f"   Sequential Time: ~{total_high_priority_time:.1f} minutes")
print(f"   Parallel Time (3 workers): ~{total_high_priority_time/3:.1f} minutes")

print(f"\n🟡 Medium Priority Tables Available: {len(MEDIUM_PRIORITY_TABLES)} tables")

🔴 High Priority Tables (Critical for POC):
   📋 AMX_DONATION_LINES          27,099 rows, 26 cols, 320.0MB (~ 0.5min)
   📋 AMX_DONATION_HEADER          1,389 rows, 30 cols,  59.0MB (~ 0.0min)
   📋 RW_ORDER_ITEM              230,282 rows, 75 cols,  72.0MB (~ 4.0min)
   📋 RW_ORDER_SUPPLIER           96,552 rows,  7 cols,   7.0MB (~ 1.7min)
   📋 RW_PURCHASE_ORDER           96,552 rows, 51 cols,  22.0MB (~ 1.7min)

📊 High Priority Summary:
   Total Rows: 451,874
   Sequential Time: ~7.9 minutes
   Parallel Time (3 workers): ~2.6 minutes

🟡 Medium Priority Tables Available: 4 tables


In [3]:
# 🔧 Section 3: Optimized Oracle Connection Manager

class OptimizedOracleExtractor:
    def __init__(self):
        # Oracle connection parameters
        self.host = os.getenv('CHOICE_ORACLE_HOST')
        self.port = os.getenv('CHOICE_ORACLE_PORT', '1521')
        self.service = os.getenv('CHOICE_ORACLE_SERVICE_NAME')
        self.user = os.getenv('CHOICE_USERNAME')
        self.password = os.getenv('CHOICE_PASSWORD')
        
        # Optimal settings based on performance testing
        self.extraction_method = 'sequential'  # Best performance: 991.8 rows/sec
        self.max_workers = 3  # For multi-table parallel processing
        self.connection_timeout = 30
        self.query_timeout = 300  # 5 minutes for large tables
        
        # Progress tracking
        self.extracted_tables = []
        self.failed_tables = []
        self.start_time = None
        self.total_rows_extracted = 0
        
        # Setup logging
        self.setup_logging()
        
        print("✅ OptimizedOracleExtractor initialized")
        print(f"🔗 Target: {self.host}:{self.port}/{self.service}")
        print(f"⚡ Method: {self.extraction_method} (optimal performance)")
        
    def setup_logging(self):
        """Setup comprehensive logging"""
        log_file = output_dirs['logs'] / f'extraction_{datetime.now().strftime("%Y%m%d_%H%M%S")}.log'
        
        logging.basicConfig(
            level=logging.INFO,
            format='%(asctime)s - %(levelname)s - %(message)s',
            handlers=[
                logging.FileHandler(log_file),
                logging.StreamHandler()
            ]
        )
        self.logger = logging.getLogger(__name__)
        self.logger.info(f"Logging initialized: {log_file}")
        
    def get_connection(self):
        """Get optimized Oracle connection"""
        try:
            dsn = cx_Oracle.makedsn(self.host, self.port, service_name=self.service)
            connection = cx_Oracle.connect(
                self.user, 
                self.password, 
                dsn,
                encoding='UTF-8'
            )
            
            # Apply Oracle optimizations based on our testing
            cursor = connection.cursor()
            cursor.execute("ALTER SESSION SET QUERY_REWRITE_ENABLED = FALSE")
            cursor.execute("ALTER SESSION SET OPTIMIZER_MODE = FIRST_ROWS")
            cursor.close()
            
            return connection
            
        except Exception as e:
            self.logger.error(f"Connection failed: {e}")
            raise
            
    def test_connection(self):
        """Test Oracle connection and get basic info"""
        try:
            connection = self.get_connection()
            cursor = connection.cursor()
            
            # Test query
            cursor.execute("SELECT COUNT(*) FROM RWTXN_46.AMX_DONATION_LINES")
            count = cursor.fetchone()[0]
            
            cursor.close()
            connection.close()
            
            print(f"✅ Connection successful!")
            print(f"📊 Test query result: AMX_DONATION_LINES has {count:,} rows")
            return True
            
        except Exception as e:
            print(f"❌ Connection failed: {e}")
            return False

# Initialize extractor
extractor = OptimizedOracleExtractor()

# Test connection
if extractor.test_connection():
    print("🚀 Ready to begin optimal data extraction!")
else:
    print("🛑 Connection issues detected. Please check credentials and network.")

2025-08-09 21:43:40,604 - INFO - Logging initialized: optimized_extraction_output/logs/extraction_20250809_214340.log


✅ OptimizedOracleExtractor initialized
🔗 Target: 52.43.135.66:1521/staging
⚡ Method: sequential (optimal performance)
✅ Connection successful!
📊 Test query result: AMX_DONATION_LINES has 27,099 rows
🚀 Ready to begin optimal data extraction!


In [4]:
# 🚀 Section 4: Sequential Table Extraction (Optimal Method)

def extract_table_sequential(table_info, save_formats=['csv', 'parquet']):
    """Extract single table using optimal sequential method"""
    table_name = table_info['name']
    full_name = table_info['full_name']
    expected_rows = table_info['rows']
    
    print(f"\n🔄 Extracting {table_name}...")
    print(f"   📋 Full name: {full_name}")
    print(f"   📊 Expected rows: {expected_rows:,}")
    
    start_time = time.time()
    
    try:
        # Get connection
        connection = extractor.get_connection()
        
        # Optimal query based on our testing (991.8 rows/sec performance)
        query = f"SELECT * FROM {full_name} ORDER BY ROWID"
        
        # Extract with pandas (most efficient for our environment)
        df = pd.read_sql_query(query, connection)
        connection.close()
        
        duration = time.time() - start_time
        actual_rows = len(df)
        throughput = actual_rows / duration if duration > 0 else 0
        
        # Save in requested formats
        saved_files = []
        
        if 'csv' in save_formats:
            csv_path = output_dirs['csv'] / f"{table_name}.csv"
            df.to_csv(csv_path, index=False)
            saved_files.append(csv_path)
            
        if 'parquet' in save_formats:
            parquet_path = output_dirs['parquet'] / f"{table_name}.parquet"
            df.to_parquet(parquet_path, index=False)
            saved_files.append(parquet_path)
        
        # Calculate file sizes
        total_size_mb = sum(f.stat().st_size for f in saved_files) / 1024**2
        
        # Memory usage
        memory_mb = df.memory_usage(deep=True).sum() / 1024**2
        
        # Log results
        result = {
            'table_name': table_name,
            'full_name': full_name,
            'rows_extracted': actual_rows,
            'expected_rows': expected_rows,
            'columns': len(df.columns),
            'duration_seconds': round(duration, 2),
            'throughput_rows_per_sec': round(throughput, 1),
            'memory_usage_mb': round(memory_mb, 2),
            'file_size_mb': round(total_size_mb, 2),
            'saved_files': [str(f) for f in saved_files],
            'status': 'success',
            'timestamp': datetime.now().isoformat()
        }
        
        # Display results
        print(f"   ✅ Completed: {actual_rows:,} rows in {duration:.2f}s")
        print(f"   🚀 Throughput: {throughput:.1f} rows/sec")
        print(f"   💾 Memory: {memory_mb:.2f}MB, Files: {total_size_mb:.2f}MB")
        print(f"   📁 Saved to: {len(saved_files)} formats")
        
        # Row count validation
        if actual_rows != expected_rows:
            print(f"   ⚠️  Row count mismatch: expected {expected_rows:,}, got {actual_rows:,}")
        
        extractor.extracted_tables.append(result)
        extractor.total_rows_extracted += actual_rows
        
        # Force garbage collection for large tables
        del df
        gc.collect()
        
        return result
        
    except Exception as e:
        duration = time.time() - start_time
        error_result = {
            'table_name': table_name,
            'full_name': full_name,
            'error': str(e),
            'duration_seconds': round(duration, 2),
            'status': 'failed',
            'timestamp': datetime.now().isoformat()
        }
        
        print(f"   ❌ Failed: {e}")
        extractor.logger.error(f"Table extraction failed: {table_name} - {e}")
        extractor.failed_tables.append(error_result)
        
        return error_result

print("✅ Sequential extraction function ready")
print("📊 Expected performance: ~950-1000 rows/sec (based on testing)")

✅ Sequential extraction function ready
📊 Expected performance: ~950-1000 rows/sec (based on testing)


In [5]:
# 🎯 Section 5: Extract High Priority Tables (Critical POC Data)

print("🔴 ===== HIGH PRIORITY TABLE EXTRACTION =====")
print(f"📊 Extracting {len(HIGH_PRIORITY_TABLES)} critical tables for POC")
print(f"⏱️  Estimated total time: ~{total_high_priority_time:.1f} minutes")

extractor.start_time = time.time()
high_priority_results = []

# Extract each high priority table sequentially (optimal method)
for i, table_info in enumerate(HIGH_PRIORITY_TABLES, 1):
    print(f"\n{'='*60}")
    print(f"🔄 [{i}/{len(HIGH_PRIORITY_TABLES)}] {table_info['name']}")
    print(f"🎯 Priority: {table_info['business_priority'].upper()}")
    print(f"📝 Description: {table_info['description']}")
    
    result = extract_table_sequential(table_info)
    high_priority_results.append(result)
    
    # Progress update
    elapsed = time.time() - extractor.start_time
    remaining_tables = len(HIGH_PRIORITY_TABLES) - i
    avg_time_per_table = elapsed / i
    eta_minutes = (remaining_tables * avg_time_per_table) / 60
    
    print(f"\n📈 Progress: {i}/{len(HIGH_PRIORITY_TABLES)} tables completed")
    print(f"⏱️  Elapsed: {elapsed/60:.1f}min, ETA: {eta_minutes:.1f}min")
    print(f"📊 Total rows extracted: {extractor.total_rows_extracted:,}")

# Final summary
total_duration = time.time() - extractor.start_time
successful_extractions = [r for r in high_priority_results if r['status'] == 'success']
failed_extractions = [r for r in high_priority_results if r['status'] == 'failed']

print(f"\n" + "="*80)
print(f"🏁 HIGH PRIORITY EXTRACTION COMPLETED")
print(f"="*80)
print(f"⏱️  Total Duration: {total_duration/60:.2f} minutes ({total_duration:.1f} seconds)")
print(f"✅ Successful Tables: {len(successful_extractions)}/{len(HIGH_PRIORITY_TABLES)}")
print(f"❌ Failed Tables: {len(failed_extractions)}")
print(f"📊 Total Rows Extracted: {extractor.total_rows_extracted:,}")
print(f"🚀 Overall Throughput: {extractor.total_rows_extracted/total_duration:.1f} rows/sec")

if successful_extractions:
    avg_throughput = np.mean([r['throughput_rows_per_sec'] for r in successful_extractions])
    total_file_size = sum(r['file_size_mb'] for r in successful_extractions)
    print(f"📈 Average Table Throughput: {avg_throughput:.1f} rows/sec")
    print(f"💾 Total Files Size: {total_file_size:.2f} MB")

# Save extraction metadata
metadata = {
    'extraction_type': 'high_priority_tables',
    'extraction_method': 'sequential_optimal',
    'start_time': datetime.fromtimestamp(extractor.start_time).isoformat(),
    'end_time': datetime.now().isoformat(),
    'duration_seconds': total_duration,
    'duration_minutes': total_duration / 60,
    'total_rows': extractor.total_rows_extracted,
    'successful_tables': len(successful_extractions),
    'failed_tables': len(failed_extractions),
    'overall_throughput': extractor.total_rows_extracted/total_duration,
    'extraction_results': high_priority_results
}

metadata_file = output_dirs['metadata'] / 'high_priority_extraction_metadata.json'
with open(metadata_file, 'w') as f:
    json.dump(metadata, f, indent=2, default=str)

print(f"\n💾 Metadata saved to: {metadata_file}")

🔴 ===== HIGH PRIORITY TABLE EXTRACTION =====
📊 Extracting 5 critical tables for POC
⏱️  Estimated total time: ~7.9 minutes

🔄 [1/5] AMX_DONATION_LINES
🎯 Priority: CRITICAL
📝 Description: Core donation line items data

🔄 Extracting AMX_DONATION_LINES...
   📋 Full name: RWTXN_46.AMX_DONATION_LINES
   📊 Expected rows: 27,099
   ✅ Completed: 27,099 rows in 26.94s
   🚀 Throughput: 1005.7 rows/sec
   💾 Memory: 25.25MB, Files: 5.15MB
   📁 Saved to: 2 formats

📈 Progress: 1/5 tables completed
⏱️  Elapsed: 0.5min, ETA: 1.8min
📊 Total rows extracted: 27,099

🔄 [2/5] AMX_DONATION_HEADER
🎯 Priority: CRITICAL
📝 Description: Donation header/summary data

🔄 Extracting AMX_DONATION_HEADER...
   📋 Full name: RWTXN_46.AMX_DONATION_HEADER
   📊 Expected rows: 1,389
   ✅ Completed: 1,389 rows in 3.34s
   🚀 Throughput: 415.6 rows/sec
   💾 Memory: 2.14MB, Files: 0.67MB
   📁 Saved to: 2 formats

📈 Progress: 2/5 tables completed
⏱️  Elapsed: 0.5min, ETA: 0.8min
📊 Total rows extracted: 28,488

🔄 [3/5] RW_ORDER_

In [6]:
# 📋 Section 6: Data Quality Validation and Summary

print("🔍 ===== DATA QUALITY VALIDATION =====")

validation_results = []

for table_result in successful_extractions:
    table_name = table_result['table_name']
    print(f"\n📊 Validating {table_name}...")
    
    try:
        # Load the extracted data
        csv_path = output_dirs['csv'] / f"{table_name}.csv"
        df = pd.read_csv(csv_path)
        
        # Basic validation metrics
        validation = {
            'table_name': table_name,
            'file_exists': csv_path.exists(),
            'rows_in_file': len(df),
            'columns_in_file': len(df.columns),
            'null_percentages': df.isnull().mean().round(3).to_dict(),
            'duplicate_rows': df.duplicated().sum(),
            'memory_usage_mb': df.memory_usage(deep=True).sum() / 1024**2,
            'data_types': df.dtypes.astype(str).to_dict(),
            'sample_data_available': True
        }
        
        # Display key metrics
        print(f"   ✅ File: {csv_path} ({validation['rows_in_file']:,} rows)")
        print(f"   📊 Columns: {validation['columns_in_file']}")
        print(f"   🔍 Duplicates: {validation['duplicate_rows']}")
        
        # Check for high null percentages
        high_null_cols = [col for col, pct in validation['null_percentages'].items() if pct > 0.9]
        if high_null_cols:
            print(f"   ⚠️  High null columns: {high_null_cols}")
        
        # Sample of data
        print(f"   📝 Sample columns: {list(df.columns[:5])}...")
        
        validation_results.append(validation)
        
        # Clean up
        del df
        
    except Exception as e:
        print(f"   ❌ Validation error: {e}")
        validation_results.append({
            'table_name': table_name,
            'validation_error': str(e)
        })

print(f"\n✅ Validation completed for {len(validation_results)} tables")

# Save validation results
validation_file = output_dirs['metadata'] / 'data_quality_validation.json'
with open(validation_file, 'w') as f:
    json.dump(validation_results, f, indent=2, default=str)

print(f"💾 Validation results saved to: {validation_file}")

🔍 ===== DATA QUALITY VALIDATION =====

📊 Validating AMX_DONATION_LINES...
   ✅ File: optimized_extraction_output/csv/AMX_DONATION_LINES.csv (27,099 rows)
   📊 Columns: 26
   🔍 Duplicates: 0
   ⚠️  High null columns: ['CUBE', 'CODEDATETYPE']
   📝 Sample columns: ['DONATIONNUMBER', 'ITEMNUMBER', 'ITEMDESCRIPTION', 'DONATIONREASON', 'PACK']...

📊 Validating AMX_DONATION_HEADER...
   ✅ File: optimized_extraction_output/csv/AMX_DONATION_HEADER.csv (1,389 rows)
   📊 Columns: 30
   🔍 Duplicates: 0
   ⚠️  High null columns: ['DCONTACTCITYSTATEZIP', 'WCONTACTFAX', 'MAGREFNUMBER', 'HTMLFILENAME']
   📝 Sample columns: ['DONATIONNUMBER', 'DONORID', 'DONORNAME', 'DCONTACTNAME', 'DCONTACTCITYSTATEZIP']...

📊 Validating RW_ORDER_ITEM...
   ✅ File: optimized_extraction_output/csv/RW_ORDER_ITEM.csv (230,282 rows)
   📊 Columns: 75
   🔍 Duplicates: 0
   ⚠️  High null columns: ['CREATED_DATE', 'CREATED_BY', 'LAST_MODIFIED_DATE', 'LAST_MODIFIED_BY', 'ATTACHMENT_FLAG', 'NOTE_FLAG', 'CHECKOUT_ID', 'CHECKOUT_

In [7]:
# 🟡 Section 7: Optional Medium Priority Table Extraction

print("\n🟡 ===== MEDIUM PRIORITY TABLE EXTRACTION (OPTIONAL) =====")
print(f"📊 {len(MEDIUM_PRIORITY_TABLES)} additional tables available for comprehensive analysis")

# Ask user if they want to continue with medium priority tables
extract_medium_priority = input("\n🤔 Extract medium priority tables? (y/n): ").lower().strip() == 'y'

if extract_medium_priority:
    print("🔄 Extracting medium priority tables...")
    
    medium_start_time = time.time()
    medium_priority_results = []
    
    for i, table_info in enumerate(MEDIUM_PRIORITY_TABLES, 1):
        print(f"\n🔄 [{i}/{len(MEDIUM_PRIORITY_TABLES)}] {table_info['name']}")
        
        result = extract_table_sequential(table_info)
        medium_priority_results.append(result)
        
        # Progress update
        elapsed = time.time() - medium_start_time
        print(f"   📈 Medium priority progress: {i}/{len(MEDIUM_PRIORITY_TABLES)}")
        print(f"   ⏱️  Elapsed: {elapsed/60:.1f}min")
    
    medium_duration = time.time() - medium_start_time
    medium_successful = [r for r in medium_priority_results if r['status'] == 'success']
    
    print(f"\n✅ Medium priority extraction completed:")
    print(f"   ⏱️  Duration: {medium_duration/60:.2f} minutes")
    print(f"   ✅ Successful: {len(medium_successful)}/{len(MEDIUM_PRIORITY_TABLES)}")
    
    # Save medium priority metadata
    medium_metadata = {
        'extraction_type': 'medium_priority_tables',
        'duration_minutes': medium_duration / 60,
        'extraction_results': medium_priority_results
    }
    
    medium_metadata_file = output_dirs['metadata'] / 'medium_priority_extraction_metadata.json'
    with open(medium_metadata_file, 'w') as f:
        json.dump(medium_metadata, f, indent=2, default=str)
        
else:
    print("⏭️  Skipping medium priority tables - focusing on high priority data for POC")
    medium_priority_results = []

print("\n🎯 High priority extraction complete - ready for analysis!")


🟡 ===== MEDIUM PRIORITY TABLE EXTRACTION (OPTIONAL) =====
📊 4 additional tables available for comprehensive analysis
⏭️  Skipping medium priority tables - focusing on high priority data for POC

🎯 High priority extraction complete - ready for analysis!


In [8]:
# 📊 Section 8: Final Extraction Summary and Next Steps

print("\n" + "="*80)
print("🏁 HUNGERHUB ORACLE EXTRACTION SUMMARY")
print("="*80)

# Calculate totals
all_successful = [r for r in extractor.extracted_tables if r['status'] == 'success']
all_failed = [r for r in extractor.failed_tables]
total_extraction_time = time.time() - extractor.start_time if extractor.start_time else 0

print(f"\n📈 Overall Performance:")
print(f"   ⏱️  Total Time: {total_extraction_time/60:.2f} minutes")
print(f"   ✅ Successful Tables: {len(all_successful)}")
print(f"   ❌ Failed Tables: {len(all_failed)}")
print(f"   📊 Total Rows: {extractor.total_rows_extracted:,}")

if total_extraction_time > 0:
    overall_throughput = extractor.total_rows_extracted / total_extraction_time
    print(f"   🚀 Overall Throughput: {overall_throughput:.1f} rows/sec")

# File inventory
print(f"\n📁 Generated Files:")
for format_name, dir_path in output_dirs.items():
    files = list(dir_path.glob('*'))
    if files:
        total_size = sum(f.stat().st_size for f in files if f.is_file()) / 1024**2
        print(f"   📄 {format_name.upper()}: {len(files)} files ({total_size:.2f} MB)")
    else:
        print(f"   📄 {format_name.upper()}: No files")

# Performance comparison with our benchmarks
if all_successful:
    avg_throughput = np.mean([r['throughput_rows_per_sec'] for r in all_successful])
    benchmark_throughput = 991.8  # From our testing
    performance_ratio = avg_throughput / benchmark_throughput
    
    print(f"\n⚡ Performance Analysis:")
    print(f"   🎯 Target Throughput: {benchmark_throughput:.1f} rows/sec")
    print(f"   📊 Achieved Throughput: {avg_throughput:.1f} rows/sec")
    print(f"   📈 Performance Ratio: {performance_ratio:.2f}x")
    
    if performance_ratio >= 0.8:
        print(f"   ✅ Performance: Excellent (within 20% of benchmark)")
    elif performance_ratio >= 0.6:
        print(f"   🟡 Performance: Good (within 40% of benchmark)")
    else:
        print(f"   🔴 Performance: Below expectations - check network/server load")

# Next steps for POC
print(f"\n🎯 Next Steps for HungerHub POC:")
print(f"   1. 📊 Load extracted data into analysis notebook")
print(f"   2. 🔍 Perform data profiling and quality assessment")
print(f"   3. 📈 Create donation analytics and visualizations")
print(f"   4. 🎨 Build Plotly Dash dashboard with key metrics")
print(f"   5. 📋 Generate business intelligence reports")

# Ready-to-use file paths for next notebook
print(f"\n📁 Key Data Files for Analysis:")
for table_info in HIGH_PRIORITY_TABLES:
    table_name = table_info['name']
    csv_path = output_dirs['csv'] / f"{table_name}.csv"
    if csv_path.exists():
        file_size = csv_path.stat().st_size / 1024**2
        print(f"   📄 {table_name}: {csv_path} ({file_size:.1f} MB)")

print(f"\n✅ EXTRACTION COMPLETED - READY FOR ANALYTICS! 🚀")
print(f"💾 All data saved to: {output_base}")
print(f"📊 Total dataset: {extractor.total_rows_extracted:,} rows across {len(all_successful)} tables")

# Memory cleanup
gc.collect()
print(f"\n🧹 Memory cleanup completed")


🏁 HUNGERHUB ORACLE EXTRACTION SUMMARY

📈 Overall Performance:
   ⏱️  Total Time: 22.14 minutes
   ✅ Successful Tables: 5
   ❌ Failed Tables: 0
   📊 Total Rows: 451,874
   🚀 Overall Throughput: 340.1 rows/sec

📁 Generated Files:
   📄 CSV: 5 files (93.60 MB)
   📄 PARQUET: 5 files (7.92 MB)
   📄 LOGS: 3 files (0.00 MB)
   📄 METADATA: 2 files (0.02 MB)

⚡ Performance Analysis:
   🎯 Target Throughput: 991.8 rows/sec
   📊 Achieved Throughput: 973.6 rows/sec
   📈 Performance Ratio: 0.98x
   ✅ Performance: Excellent (within 20% of benchmark)

🎯 Next Steps for HungerHub POC:
   1. 📊 Load extracted data into analysis notebook
   2. 🔍 Perform data profiling and quality assessment
   3. 📈 Create donation analytics and visualizations
   4. 🎨 Build Plotly Dash dashboard with key metrics
   5. 📋 Generate business intelligence reports

📁 Key Data Files for Analysis:
   📄 AMX_DONATION_LINES: optimized_extraction_output/csv/AMX_DONATION_LINES.csv (4.3 MB)
   📄 AMX_DONATION_HEADER: optimized_extraction_